# 00 — Setup smoke test

Colab entry point. Verifies the verl install (torch -> vllm==0.8.3 -> flash-attn -> verl -> ray)
landed correctly, then runs a ~10-step GRPO smoke test on a truncated GSM8K slice with
Qwen2.5-Math-1.5B-Instruct to confirm the generate -> verify -> update loop actually runs end to end.

**Hardware note:** this targets **Colab Pro (L4/A100) first** — a 1.5B model plus vLLM's
KV cache is tight on a free-tier T4's 16 GB once FSDP training weights share the GPU.
T4 is a later size-down, not the default target. If step 4 OOMs on T4, switch `MODEL_PATH`
below to `Qwen/Qwen2.5-Math-0.5B-Instruct`.

**Assumptions worth verifying if this drifts** (verl moves fast — see `CLAUDE.md`):
- Config keys below were confirmed against `verl-project/verl@main` on 2026-07-10:
  `verl/trainer/config/{ppo_trainer,rollout/rollout,actor/actor}.yaml` and
  `examples/grpo_trainer/run_qwen3_4b_fsdp.sh`. If verl has moved on since, diff those
  files against what's used here before trusting it blindly.
- We default to the **Instruct** checkpoint (`Qwen/Qwen2.5-Math-1.5B-Instruct`), not the
  base model, because the base model doesn't reliably close with `#### <answer>`, which is
  what `verl/utils/reward_score/gsm8k.py`'s regex expects for a nonzero reward. If you swap
  to the base checkpoint, pair it with a few-shot prompt that forces the `####` format —
  otherwise `critic/score/mean` will sit at 0 for all 10 steps regardless of whether the
  loop itself is working.

## 1. Clone/pull the repo and install the verl stack

In [ ]:
import os

REPO_URL = "https://github.com/<your-org>/verifier-bottleneck-math.git"  # TODO: set this
REPO_DIR = "/content/verifier-bottleneck-math"

if not os.path.isdir(REPO_DIR):
    !git clone "{REPO_URL}" "{REPO_DIR}"
else:
    !git -C "{REPO_DIR}" pull

%cd {REPO_DIR}

In [ ]:
!bash scripts/colab_install.sh

## 2. Runtime flags

In [ ]:
import os

os.environ["VLLM_USE_V1"] = "1"

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 3. Truncated GSM8K

Preprocess GSM8K with verl's own script (`examples/data_preprocess/gsm8k.py`). The pip
package (`pip install verl==0.8.0`) only ships the importable `verl/` module, not the
`examples/` directory, so we shallow-clone `volcengine/verl` at the matching `v0.8.0` tag
just to get that one script — confirmed by inspecting the actual PyPI wheel contents, which
contain only `verl/`, `scripts/`, `tests/`. Then slice the output down to 64 train / 16 val
rows so the smoke test is fast. `trainer.total_training_steps=10` below is the hard cap
regardless of dataset size — the slice just keeps step 0 (data loading, tokenization) fast
too.

In [ ]:
!git clone --depth 1 --branch v0.8.0 https://github.com/volcengine/verl.git /content/verl_src
!python3 /content/verl_src/examples/data_preprocess/gsm8k.py --local_save_dir ~/data/gsm8k_smoke

In [ ]:
import os
import pandas as pd

gsm8k_dir = os.path.expanduser("~/data/gsm8k_smoke")
train_df = pd.read_parquet(os.path.join(gsm8k_dir, "train.parquet")).iloc[:64]
test_df = pd.read_parquet(os.path.join(gsm8k_dir, "test.parquet")).iloc[:16]

train_df.to_parquet(os.path.join(gsm8k_dir, "train_smoke.parquet"))
test_df.to_parquet(os.path.join(gsm8k_dir, "test_smoke.parquet"))
print(f"train: {len(train_df)} rows, test: {len(test_df)} rows -> {gsm8k_dir}")

## 4. Stock GRPO on Qwen2.5-Math-1.5B-Instruct

`enforce_eager=False` and `gpu_memory_utilization=0.4` per `CLAUDE.md` so vLLM's KV cache
and FSDP training weights both fit on one GPU. `tensor_model_parallel_size=1` and
`n_gpus_per_node=1` since this is a single-GPU Colab runtime.

In [ ]:
import os

os.environ["MODEL_PATH"] = os.environ.get("MODEL_PATH", "Qwen/Qwen2.5-Math-1.5B-Instruct")
gsm8k_dir = os.path.expanduser("~/data/gsm8k_smoke")
os.environ["TRAIN_FILE"] = os.path.join(gsm8k_dir, "train_smoke.parquet")
os.environ["TEST_FILE"] = os.path.join(gsm8k_dir, "test_smoke.parquet")

print("MODEL_PATH:", os.environ["MODEL_PATH"])

In [ ]:
!PYTHONUNBUFFERED=1 python3 -m verl.trainer.main_ppo \
    algorithm.adv_estimator=grpo \
    algorithm.use_kl_in_reward=False \
    data.train_files="$TRAIN_FILE" \
    data.val_files="$TEST_FILE" \
    data.train_batch_size=16 \
    data.max_prompt_length=256 \
    data.max_response_length=256 \
    data.filter_overlong_prompts=True \
    data.truncation=error \
    actor_rollout_ref.model.path="$MODEL_PATH" \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.actor.optim.lr=1e-6 \
    actor_rollout_ref.actor.ppo_mini_batch_size=16 \
    actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=2 \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_coef=0.001 \
    actor_rollout_ref.actor.kl_loss_type=low_var_kl \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.n=4 \
    actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.4 \
    actor_rollout_ref.rollout.enforce_eager=False \
    actor_rollout_ref.rollout.free_cache_engine=True \
    actor_rollout_ref.rollout.log_prob_micro_batch_size_per_gpu=2 \
    actor_rollout_ref.ref.log_prob_micro_batch_size_per_gpu=2 \
    trainer.critic_warmup=0 \
    trainer.logger=console \
    trainer.project_name=vbm_smoke_test \
    trainer.experiment_name=qwen25_math_1_5b_gsm8k_grpo \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=-1 \
    trainer.test_freq=-1 \
    trainer.val_before_train=False \
    trainer.total_epochs=5 \
    trainer.total_training_steps=10 \
    2>&1 | tee grpo_smoke_test.log

## 5. Reward over steps

Parses `critic/score/mean` out of the console logger's `step:N - key:value - ...` lines
(format confirmed in `verl/utils/logger/aggregate_logger.py::concat_dict_to_str`).

In [ ]:
import re
import matplotlib.pyplot as plt

step_re = re.compile(r"step:(\d+)")
score_re = re.compile(r"critic/score/mean:([\-0-9.eE]+)")

steps, scores = [], []
with open("grpo_smoke_test.log") as f:
    for line in f:
        step_match = step_re.search(line)
        score_match = score_re.search(line)
        if step_match and score_match:
            steps.append(int(step_match.group(1)))
            scores.append(float(score_match.group(1)))

print(f"parsed {len(steps)} steps")
for s, r in zip(steps, scores):
    print(f"step {s}: reward mean = {r}")

if steps:
    plt.plot(steps, scores, marker="o")
    plt.xlabel("step")
    plt.ylabel("critic/score/mean (GSM8K rule reward)")
    plt.title("GRPO smoke test — reward over steps")
    plt.show()
else:
    print("No steps parsed — check grpo_smoke_test.log for an early failure.")